# Compute and Consolidate Comprehensive TAD Statistics

This notebook computes genome-wide and chromosome-level statistics across all evaluated TAD caller configurations and Hi-C datasets, including:
- TAD counts and density distributions
- TAD size / length distributions
- Inter-TAD gap distributions
- Chromosomal coverage profiles
- Boundary genomic footprints across all configured border definitions
- Pairwise caller and dataset concordance (Jaccard similarity matrix)


In [ ]:
import os
from pathlib import Path

import cooler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyranges as pr
import seaborn as sns
from natsort import natsorted
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]

## Parameters


In [ ]:
tad_fname_list = snakemake.input.tad_fname_list
info_fname = snakemake.input.info_fname

outdir = Path(snakemake.output.outdir)
tad_borders = snakemake.config.get("tad_borders", {})
chromosome_list = [str(c) for c in snakemake.config.get("chromosome_list", [])]

tables_dir = outdir / "tables"
figures_dir = outdir / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

## Chromosome Sizes & Genomic Information


In [ ]:
df_info = pd.read_csv(info_fname)
chromsizes = {}

for _, row in df_info.iterrows():
    cool_path = row.get("filename", "")
    if os.path.exists(cool_path) and (
        cool_path.endswith(".cool") or cool_path.endswith(".mcool")
    ):
        try:
            c = cooler.Cooler(cool_path)
            for chrom, length in c.chromsizes.items():
                chromsizes[chrom] = int(length)
                chrom_clean = chrom.replace("chr", "")
                chromsizes[chrom_clean] = int(length)
        except Exception as e:
            print(f"Could not read chromsizes from {cool_path}: {e}")

print(f"Loaded {len(chromsizes)} chromosome entries from Hi-C info.")

## Load & Parse TAD Predictions


In [ ]:
all_tads = []
all_gaps = []
summary_rows = []
chrom_coverage_rows = []
boundary_footprint_rows = []

valid_fnames = []
for fname in natsorted(tad_fname_list):
    tmp = pd.read_csv(fname)
    if len(tmp) > 0:
        valid_fnames.append(fname)

print(f"Found {len(valid_fnames)} non-empty TAD files across callers.")

In [ ]:
# Precompute reference genome length across active chromosomes
active_chroms = set()
for fname in valid_fnames:
    tmp = pd.read_csv(fname)
    active_chroms.update(tmp["chrname"].unique())

total_genome_bp = sum(chromsizes.get(c, 0) for c in active_chroms)
if total_genome_bp == 0:
    total_genome_bp = 3_092_480_053  # default hg38

for fname in tqdm(valid_fnames):
    parts = os.path.basename(fname).split(".")
    source = parts[1]
    caller_config = parts[2]

    df = pd.read_csv(fname)
    df["source"] = source
    df["caller_config"] = caller_config
    df["length"] = df["tad_stop"] - df["tad_start"]
    all_tads.append(df)

    # 1. Inter-TAD gaps per chromosome
    run_gaps = []
    for chrom, grp in df.groupby("chrname"):
        grp_sorted = grp.sort_values("tad_start")
        starts = grp_sorted["tad_start"].values[1:]
        stops = grp_sorted["tad_stop"].values[:-1]
        gap_lens = starts - stops
        for g_start, g_stop, g_len in zip(stops, starts, gap_lens):
            entry = {
                "source": source,
                "caller_config": caller_config,
                "chrname": chrom,
                "upstream_tad_stop": int(g_start),
                "downstream_tad_start": int(g_stop),
                "gap_length": int(g_len),
                "is_gap": bool(g_len > 0),
                "is_overlap": bool(g_len < 0),
            }
            all_gaps.append(entry)
            if g_len > 0:
                run_gaps.append(g_len)

    # 2. Chromosomal domain density & coverage
    for chrom, grp in df.groupby("chrname"):
        gr_chrom = pr.PyRanges(
            grp.rename(
                columns={
                    "chrname": "Chromosome",
                    "tad_start": "Start",
                    "tad_stop": "End",
                }
            )
        ).merge()
        cov_chrom_bp = sum(gr_chrom.lengths()) if len(gr_chrom) > 0 else 0
        chrom_sz = chromsizes.get(chrom, int(grp["tad_stop"].max()))
        pct_chrom = (cov_chrom_bp / chrom_sz * 100.0) if chrom_sz > 0 else 0.0
        tads_per_mb = (len(grp) / (chrom_sz / 1e6)) if chrom_sz > 0 else 0.0
        chrom_coverage_rows.append(
            {
                "source": source,
                "caller_config": caller_config,
                "chrname": chrom,
                "tad_count": len(grp),
                "covered_bp": cov_chrom_bp,
                "chrom_size": chrom_sz,
                "coverage_pct": pct_chrom,
                "tads_per_mb": tads_per_mb,
            }
        )

    # 3. Boundary genomic footprints for each configured border definition
    for border_name, border_geom in tad_borders.items():
        in_d, out_d = border_geom[0], border_geom[1]
        border_intervals = []
        for _, r in df.iterrows():
            chr_ = r["chrname"]
            s = r["tad_start"]
            e = r["tad_stop"]
            chr_sz = chromsizes.get(chr_, int(e + out_d))
            b1_s = max(0, s - out_d)
            b1_e = min(e, s + in_d)
            if b1_e > b1_s:
                border_intervals.append(
                    {"Chromosome": chr_, "Start": b1_s, "End": b1_e}
                )
            b2_s = max(s, e - in_d)
            b2_e = min(chr_sz, e + out_d)
            if b2_e > b2_s:
                border_intervals.append(
                    {"Chromosome": chr_, "Start": b2_s, "End": b2_e}
                )
        if border_intervals:
            gr_b = pr.PyRanges(pd.DataFrame(border_intervals)).merge()
            cov_b = sum(gr_b.lengths())
        else:
            cov_b = 0
        pct_b = (cov_b / total_genome_bp * 100.0) if total_genome_bp > 0 else 0.0
        boundary_footprint_rows.append(
            {
                "source": source,
                "caller_config": caller_config,
                "border_name": border_name,
                "total_border_bp": int(cov_b),
                "percent_genome_covered": float(pct_b),
            }
        )

    # 4. Genome-wide domain summary
    gr_all = pr.PyRanges(
        df.rename(
            columns={"chrname": "Chromosome", "tad_start": "Start", "tad_stop": "End"}
        )
    ).merge()
    total_cov_bp = sum(gr_all.lengths()) if len(gr_all) > 0 else 0
    pct_cov = (total_cov_bp / total_genome_bp * 100.0) if total_genome_bp > 0 else 0.0

    lens = df["length"].values
    summary_rows.append(
        {
            "source": source,
            "caller_config": caller_config,
            "num_tads": int(len(df)),
            "mean_len_bp": float(np.mean(lens)),
            "std_len_bp": float(np.std(lens)),
            "median_len_bp": float(np.median(lens)),
            "iqr_len_bp": float(np.percentile(lens, 75) - np.percentile(lens, 25)),
            "min_len_bp": int(np.min(lens)),
            "max_len_bp": int(np.max(lens)),
            "p10_len_bp": float(np.percentile(lens, 10)),
            "p90_len_bp": float(np.percentile(lens, 90)),
            "total_tad_bp": int(np.sum(lens)),
            "covered_genome_bp": int(total_cov_bp),
            "genome_coverage_pct": float(pct_cov),
            "num_inter_tad_gaps": len(run_gaps),
            "median_gap_bp": float(np.median(run_gaps)) if run_gaps else 0.0,
            "mean_gap_bp": float(np.mean(run_gaps)) if run_gaps else 0.0,
            "total_gap_bp": int(np.sum(run_gaps)) if run_gaps else 0,
        }
    )

## Pairwise Concordance Matrix (Jaccard Index)


In [ ]:
labels = [f"{r['source']}.{r['caller_config']}" for r in summary_rows]
n = len(summary_rows)
jaccard_mat = np.ones((n, n))

for i in range(n):
    gr_i = pr.PyRanges(
        all_tads[i].rename(
            columns={"chrname": "Chromosome", "tad_start": "Start", "tad_stop": "End"}
        )
    )
    for j in range(i + 1, n):
        gr_j = pr.PyRanges(
            all_tads[j].rename(
                columns={
                    "chrname": "Chromosome",
                    "tad_start": "Start",
                    "tad_stop": "End",
                }
            )
        )
        val = gr_i.stats.jaccard(gr_j)
        jaccard_mat[i, j] = val
        jaccard_mat[j, i] = val

df_jaccard = pd.DataFrame(jaccard_mat, index=labels, columns=labels)
df_jaccard.head()

## Export Consolidated Data Tables


In [ ]:
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(tables_dir / "tad_summary_metrics.csv", index=False)

df_gaps_all = (
    pd.DataFrame(all_gaps)
    if all_gaps
    else pd.DataFrame(
        columns=[
            "source",
            "caller_config",
            "chrname",
            "upstream_tad_stop",
            "downstream_tad_start",
            "gap_length",
            "is_gap",
            "is_overlap",
        ]
    )
)
df_gaps_all.to_csv(
    tables_dir / "inter_tad_gaps.csv.gz", index=False, compression="gzip"
)

df_jaccard.to_csv(tables_dir / "tad_concordance_matrix.csv")

df_footprint = pd.DataFrame(boundary_footprint_rows)
df_footprint.to_csv(tables_dir / "boundary_genomic_footprint.csv", index=False)

df_chrom_cov = pd.DataFrame(chrom_coverage_rows)
df_chrom_cov.to_csv(tables_dir / "chromosome_coverage.csv", index=False)

print("Saved 5 consolidated data tables under tables/:")
print(
    df_summary[
        ["source", "caller_config", "num_tads", "median_len_bp", "genome_coverage_pct"]
    ]
)

## Generate Visualizations


In [ ]:
# Figure 1: TAD Counts Per Dataset / Caller
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=df_summary, x="caller_config", y="num_tads", hue="source", palette="viridis"
)
plt.title("Total TAD Count per Caller Configuration", fontsize=16, fontweight="bold")
plt.xlabel("Caller Configuration")
plt.ylabel("TAD Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(figures_dir / "tad_counts_per_dataset.pdf", bbox_inches="tight")
plt.close()

In [ ]:
# Figure 2: TAD Density Per Chromosome
if len(df_chrom_cov) > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=df_chrom_cov,
        x="chrname",
        y="tads_per_mb",
        hue="caller_config",
        palette="magma",
    )
    plt.title(
        "TAD Density Across Chromosomes (TADs per Mb)", fontsize=16, fontweight="bold"
    )
    plt.xlabel("Chromosome")
    plt.ylabel("TADs / Mb")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / "tad_density_per_chromosome.pdf", bbox_inches="tight")
    plt.close()

In [ ]:
# Figure 3: TAD Length Distributions (Log10 scale)
df_all_tads = pd.concat(all_tads, ignore_index=True) if all_tads else pd.DataFrame()
if len(df_all_tads) > 0:
    plt.figure(figsize=(10, 6))
    sns.histplot(
        data=df_all_tads,
        x="length",
        hue="caller_config",
        log_scale=True,
        element="step",
        stat="density",
        common_norm=False,
    )
    plt.title("TAD Length Distributions (log scale)", fontsize=16, fontweight="bold")
    plt.xlabel("TAD Length (bp)")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.savefig(figures_dir / "tad_length_distributions.pdf", bbox_inches="tight")
    plt.close()

In [ ]:
# Figure 4: Median TAD Lengths
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_summary,
    x="caller_config",
    y="median_len_bp",
    hue="source",
    palette="coolwarm",
)
plt.title("Median TAD Length per Caller", fontsize=16, fontweight="bold")
plt.xlabel("Caller Configuration")
plt.ylabel("Median Length (bp)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(figures_dir / "tad_median_lengths.pdf", bbox_inches="tight")
plt.close()

In [ ]:
# Figure 5: Inter-TAD Gap Distributions
df_valid_gaps = (
    df_gaps_all[df_gaps_all["is_gap"]]
    if "is_gap" in df_gaps_all.columns
    else pd.DataFrame()
)
plt.figure(figsize=(10, 5))
if len(df_valid_gaps) > 0:
    sns.histplot(
        data=df_valid_gaps,
        x="gap_length",
        hue="caller_config",
        log_scale=True,
        element="step",
        stat="density",
        common_norm=False,
    )
    plt.title("Inter-TAD Gap Size Distribution", fontsize=16, fontweight="bold")
    plt.xlabel("Gap Length (bp)")
    plt.ylabel("Density")
else:
    plt.text(
        0.5,
        0.5,
        "No inter-TAD gaps identified\n(contiguous domain partition)",
        ha="center",
        va="center",
        fontsize=16,
    )
    plt.title("Inter-TAD Gap Size Distribution", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(figures_dir / "inter_tad_gap_distributions.pdf", bbox_inches="tight")
plt.close()

In [ ]:
# Figure 6: Chromosome Coverage (%)
if len(df_chrom_cov) > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=df_chrom_cov,
        x="chrname",
        y="coverage_pct",
        hue="caller_config",
        palette="mako",
    )
    plt.title("Chromosome Coverage Spanned by TADs (%)", fontsize=16, fontweight="bold")
    plt.xlabel("Chromosome")
    plt.ylabel("% Covered by TADs")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / "chromosome_coverage.pdf", bbox_inches="tight")
    plt.close()

In [ ]:
# Figure 7: Boundary Genomic Footprints Across Geometry Definitions
if len(df_footprint) > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=df_footprint,
        x="border_name",
        y="percent_genome_covered",
        hue="caller_config",
        palette="Spectral",
    )
    plt.title(
        "Boundary Genomic Footprint (% Genome Covered)", fontsize=16, fontweight="bold"
    )
    plt.xlabel("Boundary Definition")
    plt.ylabel("% of Total Genome in Boundary")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / "boundary_genomic_footprint.pdf", bbox_inches="tight")
    plt.close()

In [ ]:
# Figure 8: Caller Concordance Clustermap (Pairwise Jaccard)
if len(df_jaccard) > 1:
    cm = sns.clustermap(
        df_jaccard,
        figsize=(9, 8),
        cmap="viridis",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        cbar_kws={"label": "Jaccard Similarity"},
    )
    cm.fig.suptitle(
        "TAD Call Concordance Across Algorithms & Parameters",
        fontsize=16,
        fontweight="bold",
        y=1.02,
    )
    cm.savefig(figures_dir / "caller_concordance_clustermap.pdf", bbox_inches="tight")
    plt.close()
else:
    plt.figure(figsize=(6, 6))
    sns.heatmap(df_jaccard, annot=True, cmap="viridis", vmin=0, vmax=1)
    plt.title("TAD Call Concordance", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(figures_dir / "caller_concordance_clustermap.pdf", bbox_inches="tight")
    plt.close()